In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import roots_legendre

from shared.common import load_config, normal_density, plot_setup, save_figure
from shared.helper_funcs import gaussian_cross_section, lattice_cross_section, rotate, rural_widths

In [ ]:
settings = load_config()
field, config = settings['field'], settings['alignment']
inp, out = Path('../input'), Path('../output')
C0 = field['log_scale_mg_m3']
width_lo, width_hi = field['width_scale_bounds']
bearing_bound = field['bearing_offset_bound_deg']
nodes, weights = roots_legendre(8)
palette = plot_setup()
labels = {('source', 'joint'): 'Source wind · joint fit', ('two', 'joint'): 'Two winds · joint fit',
          ('source', 'separate'): 'Source wind · separate fit', ('two', 'separate'): 'Two winds · separate fit'}

In [ ]:
runs = pd.read_csv(inp / 'runs.csv')
obs = pd.read_csv(inp / 'receptors.csv')
wind = pd.read_csv(inp / 'surface_winds.csv')
prior = pd.read_csv(inp / 'predictions.csv')
runs = runs[['run', 'q_g_s', 'h_m', 'stability']].merge(wind, on='run', validate='one_to_one')
samples = obs.merge(runs, on='run', validate='many_to_one')
samples['toward_deg'] = (samples['source_from_deg'] + 180) % 360
# run 29: one-station forcing when the second station has no record
samples['wind_end_m_s'] = samples['downwind_speed_m_s'].fillna(samples['source_speed_m_s'])
samples['turn_deg'] = samples['downwind_corrected_from_deg'].fillna(samples['source_from_deg'])-samples['source_from_deg']
features = ['east_m', 'north_m', 'z_m', 'q_g_s', 'h_m', 'stability', 'source_speed_m_s',
            'source_sd_deg', 'toward_deg', 'wind_end_m_s', 'turn_deg']

In [ ]:
def centerline(x, bearing, u0, u1, turn_deg, two):
    if not two:
        return np.zeros_like(x), u0
    # interpolate vector components along the projected station baseline
    station_x, _ = rotate(config['station_east_m'], config['station_north_m'], bearing)
    delta = np.deg2rad(turn_deg)
    end_u, end_v = u1*np.cos(delta), u1*np.sin(delta)
    forward = (station_x > 0) & (end_u > 0)
    # off-axis station or reversed wind in a trial fit: source-only forcing
    station_x = np.where(forward, station_x, 1.0)
    end_u, end_v = np.where(forward, end_u, u0), np.where(forward, end_v, 0.0)
    within = np.minimum(x, station_x)
    f = within[..., None]/station_x[..., None]*(1+nodes)/2
    du = end_u-u0
    slopes = end_v[..., None]*f/(u0[..., None] + du[..., None]*f)
    bend = within/2*(slopes @ weights) + (x-within)*end_v/end_u
    return bend, u0 + du*within/station_x

def widths(x, stability, sd_deg):
    sy, sz = rural_widths(x, stability)
    return np.hypot(field['angle_floor_fraction']*sy, x*np.deg2rad(sd_deg)), sz

def predict(inputs, parameters, two, lattice=False):
    east, north, z, q, h, stability, u0, sd_deg, toward, u1, turn = inputs
    bearing = toward + parameters[2]
    x, y = rotate(east, north, bearing)
    downwind = x > 0
    prediction = np.zeros_like(x)
    xx = x[downwind]
    bend, wind_x = centerline(xx, bearing[downwind], u0[downwind], u1[downwind], turn[downwind], two)
    sy, sz = widths(xx, stability[downwind], sd_deg[downwind])
    args = (y[downwind]-bend, z[downwind], h[downwind], sy*parameters[0], sz*parameters[1])
    density = lattice_cross_section(*args, config['dy_m'], config['dz_m']) if lattice else gaussian_cross_section(*args)
    prediction[downwind] = 1000*q[downwind]/wind_x*density
    return prediction

In [ ]:
def weights_by_day(frame, count_rows):
    w = 1/(frame['met_day'].nunique()*frame.groupby('met_day')['run'].transform('nunique')
           *frame.groupby('run')['radius_m'].transform('nunique'))
    if count_rows:
        w /= frame.groupby(['run', 'radius_m'])['radius_m'].transform('size')
    return w.to_numpy()

def calibrate_joint(arc, two):
    inputs = [arc[col].to_numpy() for col in features]
    w = np.sqrt(weights_by_day(arc, count_rows=True))
    target = np.log1p(arc['observed_mg_m3'].to_numpy()/C0)

    def residual(par):
        return w*(np.log1p(predict(inputs, par, two)/C0) - target)

    bounds = ([width_lo, width_lo, -bearing_bound], [width_hi, width_hi, bearing_bound])
    starts = [[1, 1, 0], [0.8, 0.8, -8], [1.8, 1, 8], [2.7, 0.7, 0], [0.8, 0.8, 15]]
    fits = [least_squares(residual, start, bounds=bounds, x_scale=[1, 1, 12], max_nfev=160) for start in starts]
    return min((fit for fit in fits if fit.success), key=lambda fit: fit.cost).x

In [ ]:
# crosswind integrals and centered angular widths on arcs capturing the core
moments = []
arc = samples.query("array == 'arc'")
for (_, radius), group in arc.groupby(['run', 'radius_m']):
    group = group.assign(angle=(group['bearing_deg']-group['toward_deg']+180) % 360-180).sort_values('angle')
    concentration = group['observed_mg_m3'].to_numpy()
    if len(group) < 5 or max(concentration[0], concentration[-1]) > config['edge_fraction']*concentration.max():
        continue
    angle = np.deg2rad(group['angle'].to_numpy())
    spacing = radius*np.diff(angle)
    quadrature = np.r_[spacing[0]/2, (spacing[1:]+spacing[:-1])/2, spacing[-1]/2]
    mass = quadrature*concentration
    center = np.dot(angle, mass)/mass.sum()
    row = group.iloc[0]
    lateral, vertical = widths(radius, row.stability, row.source_sd_deg)
    moments.append(dict(run=row.run, met_day=row.met_day, fold=row.fold, radius_m=radius,
                        integral=mass.sum(), centroid=np.rad2deg(center),
                        width_ratio=radius*np.sqrt(np.dot((angle-center)**2, mass)/mass.sum())/lateral,
                        sz=vertical, z=row.z_m, h=row.h_m, q=row.q_g_s, u0=row.source_speed_m_s,
                        u1=row.wind_end_m_s, turn=row.turn_deg, toward=row.toward_deg))
moments = pd.DataFrame(moments)

In [ ]:
def calibrate_separate(training, two):
    x, toward, u0, u1, turn = [training[col].to_numpy() for col in ['radius_m', 'toward', 'u0', 'u1', 'turn']]
    bend, wind_x = centerline(x, toward, u0, u1, turn, two)
    train = training.assign(residual_bearing=training['centroid'] - np.rad2deg(np.arctan2(bend, x)))
    by_day = train.groupby(['met_day', 'run'])[['width_ratio', 'residual_bearing']].median().groupby(level='met_day').median()
    sy = np.clip(by_day['width_ratio'].median(), width_lo, width_hi)
    offset = np.clip(by_day['residual_bearing'].median(), -bearing_bound, bearing_bound)
    w = weights_by_day(train, count_rows=False)

    def error(sz):
        d = train['sz']*sz
        integral = 1000*train['q']/wind_x*(normal_density(train['z'], train['h'], d)
                                         + normal_density(train['z'], -train['h'], d))
        return np.dot(w, (np.log(integral)-np.log(train['integral']))**2)

    sz = minimize_scalar(error, bounds=(width_lo, width_hi), method='bounded').x
    return np.array([sy, sz, offset])

In [ ]:
cols = ['obs_id', 'run', 'met_day', 'array', 'radius_m', 'bearing_deg', 'z_m', 'observed_mg_m3', 'fold']
prediction = samples[cols].copy()
prediction = prediction.merge(prior[['obs_id', 'gaussian_met', 'ca_flat', 'gaussian_briggs']], on='obs_id', validate='one_to_one')
models = {'ca_flat': 'Prior flat-wind CA', 'gaussian_met': 'Prior matched Gaussian',
          'gaussian_briggs': 'Prior fitted Briggs'}
fit_rows, agreement = [], []
for (wind_key, cal), label in labels.items():
    two = wind_key == 'two'
    gaussian, lattice = f'{wind_key}_{cal}_gaussian', f'{wind_key}_{cal}_ca'
    models.update({gaussian: f'{label} · Gaussian', lattice: f'{label} · CA'})
    for fold in sorted(samples['fold'].unique()):
        train = arc.loc[arc['fold'].ne(fold)]
        if cal == 'joint':
            params = calibrate_joint(train, two)
        else:
            params = calibrate_separate(moments.loc[moments['fold'].ne(fold)], two)
        test = samples.loc[samples['fold'].eq(fold)]
        inputs = [test[col].to_numpy() for col in features]
        rows = prediction['fold'].eq(fold)
        prediction.loc[rows, gaussian] = predict(inputs, params, two)
        prediction.loc[rows, lattice] = predict(inputs, params, two, lattice=True)
        fit_rows.append((fold, wind_key, cal, *params, train['met_day'].nunique(), train['run'].nunique()))
        gap = np.log1p(prediction.loc[rows, lattice]/C0)-np.log1p(prediction.loc[rows, gaussian]/C0)
        agreement.append((fold, wind_key, cal, np.sqrt(np.mean(gap**2))))
fits = pd.DataFrame(fit_rows, columns=['fold', 'winds', 'calibration', 'scale_y', 'scale_z', 'bearing_offset_deg', 'days', 'runs'])
agreement = pd.DataFrame(agreement, columns=['fold', 'winds', 'calibration', 'ca_gaussian_rms_log1p'])
assert agreement['ca_gaussian_rms_log1p'].max() < 0.06

In [ ]:
long = prediction.melt(id_vars=cols, value_vars=list(models), var_name='model', value_name='predicted_mg_m3')
arc_rows = long.query("array == 'arc'")
arc_keys = ['model', 'met_day', 'run', 'radius_m']
peak = arc_rows.groupby(arc_keys, as_index=False)[['observed_mg_m3', 'predicted_mg_m3']].max()
bearing = arc_rows.groupby(arc_keys)[['observed_mg_m3', 'predicted_mg_m3']].idxmax()
true_bearing = arc_rows.loc[bearing['observed_mg_m3'], 'bearing_deg'].to_numpy()
pred_bearing = arc_rows.loc[bearing['predicted_mg_m3'], 'bearing_deg'].to_numpy()
bearing['abs_bearing_error_deg'] = np.abs((true_bearing-pred_bearing+180) % 360-180)
peak = peak.merge(bearing[['abs_bearing_error_deg']].reset_index(), on=arc_keys, validate='one_to_one')
scored = pd.concat([arc_rows.assign(target='Arc receptors'), peak.assign(target='Sampled arc peaks'),
                    long.query("array == 'tower'").assign(target='Tower receptors')], ignore_index=True)
scored['sq_log1p'] = (np.log1p(scored['predicted_mg_m3']/C0)-np.log1p(scored['observed_mg_m3']/C0))**2
score_columns = ['sq_log1p', 'abs_bearing_error_deg']
days = scored.groupby(['target', 'model', 'met_day', 'run', 'radius_m'])[score_columns].mean()
days = days.groupby(level=['target', 'model', 'met_day', 'run']).mean().groupby(level=['target', 'model', 'met_day']).mean()
metrics = days.groupby(level=['target', 'model']).mean().reset_index()
metrics['rmse_log1p'] = np.sqrt(metrics.pop('sq_log1p'))
metrics['label'] = metrics['model'].map(models)

In [ ]:
rng = np.random.default_rng(settings['seed'])
contrasts = [('two_joint_ca', 'source_joint_ca'), ('two_separate_ca', 'source_separate_ca'),
             ('two_separate_ca', 'two_joint_ca'), ('two_joint_ca', 'ca_flat'),
             ('two_separate_ca', 'ca_flat'), ('two_separate_ca', 'gaussian_briggs')]
paired = []
for target, values in days.groupby(level='target'):
    values = values['sq_log1p'].droplevel('target').unstack('model')
    draw = rng.integers(0, len(values), size=(field['bootstrap_days'], len(values)))
    boot = pd.DataFrame(np.sqrt(values.to_numpy()[draw].mean(axis=1)), columns=values.columns)
    centers = np.sqrt(values.mean())
    for candidate, reference in contrasts:
        lo, hi = np.quantile(boot[candidate]-boot[reference], [0.025, 0.975])
        paired.append((target, candidate, reference, centers[candidate]-centers[reference], lo, hi))
paired = pd.DataFrame(paired, columns=['target', 'candidate', 'reference', 'delta_rmse_log1p', 'lo95', 'hi95'])

In [ ]:
fits.to_csv(out / 'fit_parameters.csv', index=False, float_format='%.7g')
agreement.to_csv(out / 'lattice_gaussian_agreement.csv', index=False, float_format='%.7g')
metrics.to_csv(out / 'metrics.csv', index=False, float_format='%.7g')
paired.to_csv(out / 'paired_days.csv', index=False, float_format='%.7g')
# both response types already indexed in d_prepare
prediction[['obs_id', *[key for key in models if key.endswith('_ca')]]].to_csv(
    out / 'ca_predictions.csv', index=False, float_format='%.6g')
targets = ['Arc receptors', 'Sampled arc peaks', 'Tower receptors']
order = ['ca_flat', 'source_joint_ca', 'two_joint_ca', 'source_separate_ca', 'two_separate_ca']
table = metrics.pivot(index='model', columns='target', values='rmse_log1p').loc[order, targets]
display(table.round(3))
display(fits.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.3, 4), sharey=True, layout='constrained')
for ax, target in zip(axes, targets):
    for i, (model, value) in enumerate(table[target].items()):
        ax.plot(value, i, marker=['o', 's', 'D', '^', 'P'][i], color=palette[i], ms=7)
    ax.set(title=target, xlabel='Held-out RMSE of log(1 + C / C₀)')
    ax.grid(axis='x', alpha=0.25)
axes[0].set_yticks(range(len(order)), [models[k] for k in order])
axes[0].invert_yaxis()
sns.despine(fig=fig)
save_figure(fig, out, 'prairie_alignment')
display(fig)
plt.close(fig)